# 08 · Reachable corridor against Shapely

**Question.** Does Fieldwork's `reachable_corridor` widget draw the corridor it says it
draws?

**Rewritten 2026-10-14.** Fieldwork changed what it draws, so this check changed with it.
The corridor used to be a constant strip around every reached street, including the streets
at the very edge of the budget — someone standing there has nothing left to walk, and the
corridor still claimed 25 m of ground around them. The walk off the street at the end of a
route is now charged: the buffer at any point is `min(bufferM, the budget left there)`, so
the corridor tapers to nothing where the time runs out. The route also now starts at the
site rather than at the junction it was snapped to. The previous version of this notebook
rebuilt a constant strip and agreed with a Fieldwork that drew one; both have changed, and
the fixture with them.

The widget declares the rule, and this rebuilds it from the declaration using GEOS through
Shapely, rather than comparing geometry feature by feature.

The fixture is the five-minute band of the Soho corridor — 134 reached junctions, a 25 m
limit on the final walk — and it carries the street geometry, because a corridor cannot be
rebuilt from endpoints: these streets curve and the buffer follows the curve.

**This reference is deliberately finer than Fieldwork.** It steps the route in 1 m pieces
and uses the exact budget left at each, where Fieldwork cuts at 12.5 m and rounds the
radius down into six steps. So the reference is the rule as stated, and Fieldwork is a
discretisation of it — which is the thing being measured.

**Criteria fixed before running, with reasons.**

1. *Entered and partial segment counts, exactly.* Both implementations select streets by
   the same stated arithmetic over the same numbers, so a difference here would be a
   difference in the rule, not in the geometry engine.
2. *Fieldwork's area at most 100.5% of the reference.* Its discretisation pulls both ways:
   rounding the radius down removes ground, while buffering each chunk separately gives the
   chunk's far end a round cap at the chunk's radius where the rule gives a smaller one.
   Neither can be large, and a corridor materially larger than the stated rule would mean
   the taper is not tracking the budget. The half per cent also absorbs the two engines and
   two projections, which is what the old 1% criterion was for.
3. *Fieldwork's area at least 85% of the reference.* A deficit beyond that would mean the
   taper is biting where the rule does not put it.
4. *Symmetric difference at most 8%.* Equal areas can be different shapes. Looser than the
   2% the constant-width check used, because the two discretisations differ along the whole
   frontier by construction rather than only in total.

In [ ]:
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_08_corridor import run

report = run(Path.cwd() / "fixtures" / "corridor-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"reached junctions: {report['reachedJunctions']}")
print(f"segments entered: Fieldwork {report['fieldwork']['enteredSegments']}, "
      f"Shapely {report['reference']['enteredSegments']}")
print(f"entered part-way: Fieldwork {report['fieldwork']['partialSegments']}, "
      f"Shapely {report['reference']['partialSegments']}")
print(f"area: Fieldwork {report['fieldwork']['areaSquareMetres']:.0f} m2, "
      f"Shapely {report['reference']['areaSquareMetres']:.0f} m2")
print(f"area difference        {report['areaDifferenceFraction']*100:.3f}%")
print(f"symmetric difference   {report['symmetricDifferenceFraction']*100:.3f}%")

## Where the residual comes from

The segment selection agrees exactly: both implementations enter the same 283 streets and
cut 38 of them part-way. That is the part of the operation that carries the method, and it
is reproduced from the written rule rather than copied.

What is left is the taper, and it is worth saying which way it goes. Fieldwork draws
**slightly more** ground than the rule, not less: rounding each chunk's radius down takes
ground away, but buffering each chunk on its own gives its far end a round cap at that
chunk's radius where the rule gives a smaller one, and on this corridor the second effect
is the larger. The excess is a sixth of a per cent, and the symmetric difference is the
same size, which says it is spread around the boundary rather than a piece of corridor
present in one and missing from the other.

So the approximation is not purely conservative. That is recorded here and in the widget's
own source rather than assumed away.

## What agreement here does not mean

Both implementations trust the same street geometry and the same lengths, so a wrong
network is wrong in both and this check still passes.

And neither says anything about the 25 m. It is now a modelled quantity — the furthest
anyone is assumed to walk off a street at the end of a route — which makes it a claim about
people and buildings rather than a drawing width, and no reasoning for the number is
recorded anywhere in the case study it came from. Within it, the corridor still counts the
insides of buildings, locked yards and the far side of walls as reached.

The cell below asks what that unrecorded number is worth, by rebuilding the rule at other
values with everything else held fixed.

In [ ]:
# What the unrecorded 25 m is worth, with everything else held fixed: the rule rebuilt at
# each limit on the final walk. This is the reference implementation answering the question,
# not Fieldwork. Since 2026-10-14 the limit is charged, so a larger one does not simply
# widen the strip -- it can only use budget that is actually left, which is why the growth
# falls away at the top of the range.
import numpy as np, pyproj, shapely
from shapely.geometry import LineString
from shapely.ops import substring, unary_union
from checks.check_08_corridor import tapered

fixture = json.loads((Path.cwd() / "fixtures" / "corridor-soho-001.json").read_text())
declared = fixture["declaredParameters"]
to_metric = pyproj.Transformer.from_crs("OGC:CRS84", declared["metricCRS"], always_xy=True)
project = lambda g: shapely.transform(
    g, lambda c: np.column_stack(to_metric.transform(c[:, 0], c[:, 1])))

nodes = {n["id"]: n["coordinates"] for n in fixture["input"]["network"]["nodes"]}
remaining = {r["networkNodeId"]: r["remainingM"] for r in fixture["input"]["reached"]}
origin = fixture["input"].get("origin")

def corridor(final_leg_m):
    pieces = []
    if origin:
        entry = project(shapely.Point(nodes[origin["networkNodeId"]]))
        site = project(shapely.Point(origin["coordinates"]))
        pieces += tapered(LineString([site, entry]),
                          remaining[origin["networkNodeId"]] + origin["snapDistanceM"],
                          final_leg_m, declared["quadrantSegments"])
    for edge in fixture["input"]["network"]["edges"]:
        left = remaining.get(edge["from"])
        if left is None or left <= 0:
            continue
        coords = edge.get("geometry") or [nodes[edge["from"]], nodes[edge["to"]]]
        line = project(LineString(coords))
        start = project(shapely.Point(nodes[edge["from"]]))
        if shapely.Point(line.coords[0]).distance(start) > shapely.Point(line.coords[-1]).distance(start):
            line = LineString(list(line.coords)[::-1])
        piece = substring(line, 0.0, min(1.0, left / edge["lengthM"]), normalized=True)
        if not piece.is_empty and piece.length > 0:
            pieces += tapered(piece, left, final_leg_m, declared["quadrantSegments"])
    return unary_union(pieces).area

base = corridor(declared["bufferM"])
print(f"{'final walk m':>13} {'area m2':>10} {'vs 25 m':>9}")
for limit in (10, 15, 25, 40, 60):
    area = corridor(limit)
    marker = "  <- the case study's figure, with no reason recorded" if limit == declared["bufferM"] else ""
    print(f"{limit:13d} {area:10.0f} {area/base:8.2f}x{marker}")